# Evaluation Results Tables

This notebook loads saved outputs from `src/evaluate.py` and builds latex tables across all benchmarks:

- RMSE table with In-domain and Out-domain values
- All-metrics table (RMSE, nRMSE, Max, Boundary, Conserved, Fourier)

First we set the working directory, load necessary libraries and addd some helper functions.

In [ ]:
import re
from pathlib import Path

import pandas as pd
import numpy as np
import os
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if (PROJECT_ROOT / "src").exists() is False:
    PROJECT_ROOT = Path.cwd().resolve()

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
BENCHMARKS = [
    {"key": "advection", "label": "Advection", "experiment_name": "advection_benchmark"},
    {"key": "burgers", "label": "Burgers", "experiment_name": "burgers_benchmark"},
    {"key": "reactiondiffusion1d", "label": "Reaction-Diffusion 1D", "experiment_name": "reactiondiffusion1d_benchmark"},
    {"key": "reactiondiffusion2d", "label": "Reaction-Diffusion 2D", "experiment_name": "reactiondiffusion2d_benchmark"},
]

MODEL_ORDER = ["fno", "cape_fno", "late_fusion"]
MODEL_DISPLAY = {
    "fno": "FNO",
    "cape_fno": "CAPE-FNO",
    "late_fusion": "Late Fusion",
}

def eval_dir(experiment_name: str) -> Path:
    return PROJECT_ROOT / "outputs" / experiment_name / "evaluation"

def sci_3sig(x: float) -> str:
    if pd.isna(x):
        return "-"
    s = f"{x:.2e}"
    return re.sub(r"e([+-])0*(\d+)", r"e\1\2", s)

def pm_str(mean: float, std: float) -> str:
    if pd.isna(mean):
        return "-"
    if pd.isna(std):
        std = 0.0
    return f"{sci_3sig(mean)}$\\pm${sci_3sig(std)}"


def loader_domain(loader_name: str) -> str:
    return "id" if "id" in str(loader_name).lower() else "od"

We load computed errors generated using evaluate.py.

In [ ]:
# Load RMSE summaries from evaluate.py outputs
rmse_summary_by_benchmark = {}
missing_rmse = []

for b in BENCHMARKS:
    summary_path = eval_dir(b["experiment_name"]) / "test_summary_by_loader.csv"
    if not summary_path.exists():
        missing_rmse.append((b["label"], str(summary_path)))
        continue

    df = pd.read_csv(summary_path)
    if "domain" not in df.columns:
        df["domain"] = df["loader"].map(loader_domain)

    rmse_summary_by_benchmark[b["key"]] = {
        "label": b["label"],
        "df": df,
        "path": summary_path,
    }

print(f"Loaded RMSE summaries for {len(rmse_summary_by_benchmark)} / {len(BENCHMARKS)} benchmarks")
if missing_rmse:
    print("Missing RMSE summary files:")
    for label, path in missing_rmse:
        print(f"- {label}: {path}")

We create a combined rmse table Benchmark x Model with In-domain and Out-domain

In [ ]:
rmse_rows = []

for b in BENCHMARKS:
    key = b["key"]
    if key not in rmse_summary_by_benchmark:
        continue

    d = rmse_summary_by_benchmark[key]["df"].copy()
    d["model_key"] = d["model"].astype(str).str.strip().str.lower().str.replace("-", "_", regex=False)

    for model_key in MODEL_ORDER:
        d_model = d[d["model_key"] == model_key]

        id_row = d_model[d_model["domain"].str.lower() == "id"]
        od_row = d_model[d_model["domain"].str.lower() == "od"]

        id_mean = id_row["test_rmse_mean"].iloc[0] if not id_row.empty else np.nan
        id_std = id_row["test_rmse_std"].iloc[0] if not id_row.empty else np.nan
        od_mean = od_row["test_rmse_mean"].iloc[0] if not od_row.empty else np.nan
        od_std = od_row["test_rmse_std"].iloc[0] if not od_row.empty else np.nan

        rmse_rows.append(
            {
                "Benchmark": b["key"],
                "Model": model_key,
                "In-domain": pm_str(id_mean, id_std),
                "Out-domain": pm_str(od_mean, od_std),
            }
        )

rmse_summary = pd.DataFrame(rmse_rows)
rmse_summary

We create a latex table from the summary table.

In [ ]:
# LaTeX for RMSE table in the requested grouped style
import importlib
import src.utils.evaluation.table_maker as table_maker

importlib.reload(table_maker)
rmse_summary_to_grouped_latex = table_maker.rmse_summary_to_grouped_latex

rmse_table_latex = rmse_summary_to_grouped_latex(rmse_summary)
print(rmse_table_latex)

Load all metrics computed using evaluate.py.

In [ ]:
# Load all-metrics outputs from evaluate.py (requires --save-all-metrics)
all_metrics_by_benchmark = {}
missing_all_metrics = []

for b in BENCHMARKS:
    metrics_path = eval_dir(b["experiment_name"]) / "test_all_metrics_per_selected_run.csv"
    if not metrics_path.exists():
        missing_all_metrics.append((b["label"], str(metrics_path)))
        continue

    df = pd.read_csv(metrics_path)
    if "domain" not in df.columns and "loader" in df.columns:
        df["domain"] = df["loader"].map(loader_domain)

    all_metrics_by_benchmark[b["key"]] = {
        "label": b["label"],
        "df": df,
        "path": metrics_path,
    }

print(f"Loaded all-metrics files for {len(all_metrics_by_benchmark)} / {len(BENCHMARKS)} benchmarks")
if missing_all_metrics:
    print("Missing all-metrics files:")
    for label, path in missing_all_metrics:
        print(f"- {label}: {path}")

We create a combined all-metrics table accross benchmarks.

In [ ]:
def build_metrics_table_for_benchmark(df: pd.DataFrame, benchmark_label: str) -> pd.DataFrame:
    id_cols = {"run_name", "model", "seed", "domain", "loader", "combination"}
    metric_cols = [c for c in df.columns if c not in id_cols]

    if not metric_cols:
        return pd.DataFrame()

    work = df.copy()
    work["model_key"] = work["model"].astype(str).str.strip().str.lower().str.replace("-", "_", regex=False)
    work["domain_key"] = work["domain"].astype(str).str.strip().str.lower()

    agg = (
        work.groupby(["model_key", "domain_key"], as_index=False)[metric_cols]
        .agg(["mean", "std"])
    )
    agg.columns = ["_".join([p for p in col if p]).rstrip("_") for col in agg.columns.to_flat_index()]

    rows = []
    for metric in metric_cols:
        row = {"Benchmark": benchmark_label, "Metric": metric}
        for mk in MODEL_ORDER:
            for dk, dk_name in [("id", "ID"), ("od", "OD")]:
                col_name = f"{MODEL_DISPLAY[mk]} {dk_name}"
                subset = agg[(agg["model_key"] == mk) & (agg["domain_key"] == dk)]
                if subset.empty:
                    row[col_name] = "-"
                    continue

                m_col = f"{metric}_mean"
                s_col = f"{metric}_std"
                if m_col not in subset.columns:
                    row[col_name] = "-"
                    continue

                mean_val = subset[m_col].iloc[0]
                std_val = subset[s_col].iloc[0] if s_col in subset.columns else np.nan
                row[col_name] = pm_str(mean_val, std_val)

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# Combined all-metrics table across benchmarks
all_metrics_tables = []
for b in BENCHMARKS:
    key = b["key"]
    if key not in all_metrics_by_benchmark:
        continue
    d = all_metrics_by_benchmark[key]["df"]
    all_metrics_tables.append(build_metrics_table_for_benchmark(d, b["label"]))

all_metrics_table = pd.concat(all_metrics_tables, ignore_index=True) if all_metrics_tables else pd.DataFrame()
all_metrics_table

Create latex table from all-metrics summary dataframe.

In [ ]:
# LaTeX for all-metrics table in the requested grouped style
import importlib
import src.utils.evaluation.table_maker as table_maker

importlib.reload(table_maker)
all_metrics_summary_to_grouped_latex = table_maker.all_metrics_summary_to_grouped_latex

all_metrics_table_latex = all_metrics_summary_to_grouped_latex(all_metrics_table)
print(all_metrics_table_latex)